# Every picture, and why it exists

A graph you cannot see is a graph you have to take on trust. This notebook draws
each kind of picture the library makes, on a real job, and says what each one
would hide if you drew it the flat way instead.

Seven pictures, seven different questions:

| picture | the question |
|---|---|
| **shape** | what runs when, and what runs together? |
| **route space** | what am I choosing between? |
| **funnel** | how much of that did I actually look at? |
| **evidence** | which step is the problem? |
| **timeline** | what actually happened, and did it really run in parallel? |
| **scoreboard** | what did the winning route beat? |
| **figure** | the same thing, for a document that is not a web page |

Nothing here needs a browser, a dataset or a model. It all runs in a few
seconds on a bare machine.

In [1]:
try:
    import browsergraph  # noqa: F401
except ImportError:
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import json, pathlib
from dataclasses import replace

from browsergraph import execute, viz
from browsergraph.compile import compile_route
from browsergraph.manifest import NodeManifest, PortSpec
from browsergraph.workbench import Edge, NodeCandidate, StageDefinition, WorkbenchDefinition

# A fresh folder each run. Left-over files from a previous run make the "what
# did this produce" list a lie, and that list is half the point here.
import shutil
WORK = pathlib.Path("work")
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

# These come from the library rather than being redefined in every notebook.
# They used to be thirty lines pasted into each one, which meant anyone copying
# a notebook to start a project got helpers that did not exist in browsergraph.
from browsergraph.quick import chain, fanin, fanout, link, node, problems, step
from browsergraph.quick import graph as _graph
from browsergraph.quick import subgraph  # noqa: F401  (used by later notebooks)

# The notebooks kept the older names, and `build` also prints what is wrong
# rather than raising — in a notebook the complaint is the lesson.
stage = step

def build(title, task, stages, nodes, edges=()):
    bench = _graph(title, task, stages, nodes, edges)
    print("problems:", problems(bench) or "none")
    return bench

print("ready")

ready


## The job

A support-ticket pipeline. It reads tickets, then does two independent things —
works out the topic, and works out the urgency — and those meet at a routing
decision. Then it either escalates or files, which is a branch: only one of
those ever runs.

Three ways to do most steps, so there is something to choose between.

In [2]:
from browsergraph.quick import fanin, fanout, graph, link, node, passthrough, step
from browsergraph.workbench import OptimizationObjective, OptimizationProfile

nodes = [
    node("read.jsonl",  "read",   gives=[("out", "Tickets")]),
    node("read.csv",    "read",   gives=[("out", "Tickets")]),

    node("topic.keywords", "topic", [("in", "Tickets")], [("out", "Topics")]),
    node("topic.bagofwords","topic", [("in", "Tickets")], [("out", "Topics")]),
    node("topic.embed",    "topic", [("in", "Tickets")], [("out", "Topics")],
         deterministic=False),

    node("urgency.rules",  "urgency", [("in", "Tickets")], [("out", "Urgency")]),
    node("urgency.model",  "urgency", [("in", "Tickets")], [("out", "Urgency")],
         deterministic=False),

    # Doing nothing is a candidate. "Enrich the ticket" is a real obligation
    # even when there is nothing to add, and deleting the step would make two
    # routes incomparable rather than comparable.
    passthrough("enrich.none", "enrich", "Topics"),
    node("enrich.history",  "enrich", [("in", "Topics")], [("out", "Topics")]),

    node("route.decide", "route", [("topic", "Topics"), ("urgency", "Urgency")],
         [("escalate", "Ticket"), ("file", "Ticket")]),
    node("escalate.page", "escalate", [("in", "Ticket")], [("out", "Receipt")],
         effects=("notify.person",)),
    node("file.queue",    "file",     [("in", "Ticket")], [("out", "Receipt")]),
]

steps = [
    step("read",    "Read the tickets", [], [("out", "Tickets")], "read",
         ["read.jsonl", "read.csv"]),
    step("topic",   "Work out the topic", [("in", "Tickets")], [("out", "Topics")],
         "topic", ["topic.keywords", "topic.bagofwords", "topic.embed"]),
    step("urgency", "Work out the urgency", [("in", "Tickets")], [("out", "Urgency")],
         "urgency", ["urgency.rules", "urgency.model"]),
    step("enrich",  "Add what we know", [("in", "Topics")], [("out", "Topics")],
         "enrich", ["enrich.none", "enrich.history"]),
    step("route",   "Escalate or file?",
         [("topic", "Topics"), ("urgency", "Urgency")],
         [("escalate", "Ticket"), ("file", "Ticket")], "route",
         ["route.decide"], kind="branch"),
    step("escalate","Page somebody", [("in", "Ticket")], [("out", "Receipt")],
         "escalate", ["escalate.page"]),
    step("file",    "Put it in the queue", [("in", "Ticket")], [("out", "Receipt")],
         "file", ["file.queue"]),
]

links = [*fanout("read", ["topic", "urgency"]),
         link("topic", "enrich"),
         # A join has to say which port each arrival lands on. `fanin` takes
         # that mapping, so the wiring cannot be written without it.
         *fanin({"enrich": "topic", "urgency": "urgency"}, "route"),
         link("route", "escalate", from_port="escalate"),
         link("route", "file", from_port="file")]

bench = graph("Triage a ticket",
              "Read tickets, judge topic and urgency, then escalate or file.",
              steps, nodes, links,
              profiles=[OptimizationProfile(id="p", objectives=(
                  OptimizationObjective("quality", "maximize", 1.0),))])

print("problems:    ", bench.validate() or "none")
print("layers:      ", bench.layers())
print("routes:      ", bench.route_count())
print("computations:", bench.computation_count())

problems:     none
layers:       [['read'], ['topic', 'urgency'], ['enrich'], ['route'], ['escalate', 'file']]
routes:       24
computations: 48


Two counts, because there are two questions.

**Routes — 24.** How many plans you could build: one candidate for each of the
seven steps, multiplied out. That includes naming a candidate for `file` even on
a run that escalates, because the plan is fixed before anyone knows which way
the branch will go.

**Computations — 48.** How many different things this graph can be seen doing.
Each of the 24 plans can escalate or it can file, and those are not the same
thing happening.

Which is bigger depends on the graph, so it is worth having both. Put four ways
of escalating and three ways of filing behind that branch and it flips: twelve
plans, but only seven behaviours, because plans differing solely behind the side
that was not taken do exactly the same thing.

The one printed next to a search has to be **routes**, because that is what the
search can reach. An earlier version printed the other here and had `solve`
announce a champion "out of 48 possible" for a space it could only ever draw 24
routes from. Drawing this notebook is what caught it.

---

## 1 · Shape — what runs when

Position is meaning. Two boxes in the same layer are genuinely independent and
may run at once; that is derived from which ports feed which, not asserted by
whoever drew it.

In [3]:
viz.dag(bench)

Figure(svg='<svg viewBox="0 0 1100 308" width="1100" height="308" style="max-width:none" role="img"><defs><marker id="bg28544655-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Read the tickets</text><text x="69" y="149.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="363.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1 · 2 parallel</text><g><rect x="270" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="279" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Work out the topic</text><text x="279" y="108.0" font-size="9.5" fill="#68737f">3 candidates</text></g><g><rect x="270" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="279" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Work out the urgency</text><text x="279" y="190.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="573.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2</text><g><rect x="480" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="489" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Add what we know</text><text x="489" y="149.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="783.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 3</text><g><rect x="690" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1" stroke-dasharray="6 3"/><text x="699" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Escalate or file?</text><text x="867" y="134.0" text-anchor="end" font-size="9" font-weight="700" fill="#2d6cb5">BRANCH</text><text x="699" y="149.0" font-size="9.5" fill="#68737f">1 candidate · one way out</text></g><text x="993.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 4 · 2 parallel</text><g><rect x="900" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="909" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Page somebody</text><text x="909" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><g><rect x="900" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="909" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Put it in the queue</text><text x="909" y="190.0" font-size="9.5" fill="#68737f">1 candidate</text></g><path d="M246,141.0 C258.0,141.0 258.0,100.0 270,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg28544655-arrow)"/><path d="M246,141.0 C258.0,141.0 258.0,182.0 270,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg28544655-arrow)"/><path d="M456,100.0 C468.0,100.0 468.0,141.0 480,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg28544655-arrow)"/><path d="M666,141.0 C678.0,141.0 678.0,141.0 690,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg28544655-arrow)"/><text x="678.0" y="136.0" text-anchor="middle" font-size="9" fill="#68737f">topic</text><path d="M456,182.0 C573.0,182.0 573.0,141.0 690,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg28544655-arrow)"/><text x="573.0" y="156.5" text-anchor="middle" font-size="9" fill="#68737f">urgency</text><path d="M876,141.0 C888.0,141.0 888.0,100.0 900,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg28544655-arrow)"/><text x="888.0" y

Three things this picture says that a numbered list of steps cannot.

**`topic` and `urgency` sit side by side**, so they are independent — they can
run together and they fail separately.

**`route` has a dashed outline** and says BRANCH: one way out is taken and the
other path is skipped, not failed.

**Two arrows arrive at `route`, each labelled with the port it lands on.** An
unlabelled join is the bug this library was written to stop: two things arriving
at one step with no way to see which input each one feeds.

## 2 · Route space — what am I choosing between

One column per step, one box per option, one faint line per route. The red line
is a route chosen after evidence; the dashed amber one is what got chosen with
none.

In [4]:
before = {"read": "read.csv", "topic": "topic.keywords", "urgency": "urgency.rules",
          "enrich": "enrich.none", "route": "route.decide",
          "escalate": "escalate.page", "file": "file.queue"}
after  = dict(before, read="read.jsonl", topic="topic.embed",
              enrich="enrich.history")

viz.route_space(bench, route=after, alternative=before)

Figure(svg='<svg viewBox="0 0 1180 260" width="1180" height="260" style="max-width:none" role="img"><style>.bg90376722-v{cursor:pointer}.bg90376722-v:hover rect{stroke:#c0392b;stroke-width:2}</style><polyline points="46,96 170,96 206,96 330,96 366,96 490,96 526,96 650,96 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 206,96 330,96 366,96 490,96 526,96 650,96 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 206,126 330,126 366,96 490,96 526,96 650,96 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 206,126 330,126 366,96 490,96 526,96 650,96 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 206,156 330,156 366,96 490,96 526,96 650,96 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 206,156 330,156 366,96 490,96 526,96 650,96 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 206,96 330,96 366,126 490,126 526,96 650,96 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 206,96 330,96 366,126 490,126 526,96 650,96 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 206,126 330,126 366,126 490,126 526,96 650,96 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 206,126 330,126 366,126 490,126 526,96 650,96 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 206,156 330,156 366,126 490,126 526,96 650,96 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 206,156 330,156 366,126 490,126 526,96 650,96 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 206,96 330,96 366,96 490,96 526,126 650,126 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 206,96 330,96 366,96 490,96 526,126 650,126 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 206,126 330,126 366,96 490,96 526,126 650,126 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 206,126 330,126 366,96 490,96 526,126 650,126 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 206,156 330,156 366,96 490,96 526,126 650,126 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 206,156 330,156 366,96 490,96 526,126 650,126 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 206,96 330,96 366,126 490,126 526,126 650,126 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 206,96 330,96 366,126 490,126 526,126 650,126 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 206,126 330,126 366,126 490,126 526,126 650,126 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#

The picture is the *difference* between two routes, which is why two are drawn.
One highlighted path would show a fixed pipeline — the exact thing this design
argues against.

Three of the seven steps changed. The other four are the same in both, and you
can see that at a glance instead of diffing two dictionaries.

## 3 · Funnel — how much of it did I look at

The honest counter. Every number in it comes from a real search — none of them
are typed in, which is the only thing that makes a funnel worth drawing.

Bar length is log-scaled and the caption says so. A funnel from thousands down
to one is four invisible slivers on a linear axis, and a chart nobody can read
is a chart that can claim whatever it likes.

In [5]:
from browsergraph import search

# A deliberately small budget, so the picture has something to show. Given more
# evaluations than the space has routes, a search simply enumerates it and the
# funnel is two bars of the same length.
found = search.within(bench, bench.optimization_profiles[0], evaluations=6, seed=1)

viz.funnel([
    ("every route",     found.total),
    ("policy-eligible", found.eligible_total),
    ("scored",          found.examined),
    ("chosen",          1),
], title=f"what a {found.strategy} search with 6 evaluations really saw")

Figure(svg='<svg viewBox="0 0 1000 296" width="1000" height="296" style="max-width:none" role="img"><text x="176" y="83" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">every route</text><rect x="190" y="66" width="670.0" height="26" rx="4" fill="#2d6cb5" opacity="0.72" stroke="#2d6cb5" stroke-width="1"/><text x="870.0" y="83" font-size="11" fill="#22303f">24</text><text x="176" y="129" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">policy-eligible</text><rect x="190" y="112" width="670.0" height="26" rx="4" fill="#2d6cb5" opacity="0.72" stroke="#2d6cb5" stroke-width="1"/><text x="870.0" y="129" font-size="11" fill="#22303f">24</text><text x="934.0" y="129" font-size="10" fill="#68737f">÷1</text><text x="176" y="175" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">scored</text><rect x="190" y="158" width="533.9" height="26" rx="4" fill="#2d6cb5" opacity="0.62" stroke="#2d6cb5" stroke-width="1"/><text x="733.9" y="175" font-size="11" fill="#22303f">12</text><text x="797.9" y="175" font-size="10" fill="#68737f">÷2</text><text x="176" y="221" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">chosen</text><rect x="190" y="204" width="144.3" height="26" rx="4" fill="#1f8a4c" opacity="0.33" stroke="#1f8a4c" stroke-width="1"/><text x="344.3" y="221" font-size="11" fill="#22303f">1</text><text x="408.3" y="221" font-size="10" fill="#68737f">÷12</text><text x="190" y="278" font-size="9.5" fill="#68737f">bar length is log-scaled; labels are exact counts</text></svg>', title='what a greedy search with 6 evaluations really saw', note='Every row is a real filter, in order.', width=1000, height=296)

A search that scored a handful of routes and then announces "the best route",
without saying how many, is making a claim it did not earn. The number costs
nothing to carry — the search already knows it.

## 4 · Timeline — what actually happened

Everything so far has been about the graph. This one is about a run.

The shape picture says `topic` and `urgency` *may* run together. Whether they
*did* is a different question, and the only way to answer it is to look at where
the bars sit. So let us run it — twice, on one worker and then on two — and put
the two pictures side by side.

In [6]:
import time

def slow(work):
    """Each step takes a beat, so the bars have something to show."""
    def do(**kw):
        time.sleep(0.12)
        return work(**kw)
    return do

runtime = execute.Runtime({
    "read.jsonl":  slow(lambda **kw: [{"id": 1, "text": "cannot log in"},
                                      {"id": 2, "text": "invoice wrong"}]),
    "read.csv":    slow(lambda **kw: []),
    "topic.keywords":  slow(lambda **kw: ["login", "billing"]),
    "topic.bagofwords":slow(lambda **kw: ["login", "billing"]),
    "topic.embed":     slow(lambda **kw: ["access", "billing"]),
    "urgency.rules":   slow(lambda **kw: ["high", "low"]),
    "urgency.model":   slow(lambda **kw: ["high", "low"]),
    "enrich.none":     lambda **kw: kw["in"],
    "enrich.history":  slow(lambda **kw: list(kw["in"]) + ["seen before"]),
    # A branch node names the one port it chose: (port, value). The library
    # marks every other port "not taken", which is how the steps behind them
    # end up skipped instead of running on an empty value.
    "route.decide":    slow(lambda **kw: ("escalate", {"id": 1})),
    "escalate.page":   slow(lambda **kw: "paged the on-call engineer"),
    "file.queue":      slow(lambda **kw: "queued"),
})

plan = compile_route(bench, after)
one = execute.run(plan, runtime, workers=1)
two = execute.run(plan, runtime, workers=2)

print(f"one worker : {one.seconds:.2f}s   ok={one.ok}")
print(f"two workers: {two.seconds:.2f}s   ok={two.ok}")
viz.timeline(one, title="one worker — the same plan, run sequentially")

one worker : 0.72s   ok=True
two workers: 0.60s   ok=True


Figure(svg='<svg viewBox="0 0 1000 336" width="1000" height="336" style="max-width:none" role="img"><text x="190" y="34" font-size="10.5" fill="#68737f">0s</text><text x="870" y="34" text-anchor="end" font-size="10.5" fill="#68737f">0.721s</text><line x1="190" y1="42" x2="870" y2="42" stroke="#dfe5ec" stroke-width="1"/><text x="176" y="90" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">read</text><rect x="190.1" y="77" width="113.3" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>read.jsonl — ran, 120.2ms</title></rect><text x="312.4" y="91" font-size="10" fill="#68737f">120ms · ran</text><text x="176" y="120" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">topic</text><rect x="303.4" y="107" width="113.3" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>topic.embed — ran, 120.2ms</title></rect><text x="425.8" y="121" font-size="10" fill="#68737f">120ms · ran</text><text x="176" y="150" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">urgency</text><rect x="416.8" y="137" width="113.3" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>urgency.rules — ran, 120.2ms</title></rect><text x="539.0" y="151" font-size="10" fill="#68737f">120ms · ran</text><text x="176" y="180" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">enrich</text><rect x="530.1" y="167" width="113.3" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>enrich.history — ran, 120.2ms</title></rect><text x="652.4" y="181" font-size="10" fill="#68737f">120ms · ran</text><text x="176" y="210" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">route</text><rect x="643.4" y="197" width="113.3" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>route.decide — ran, 120.2ms</title></rect><text x="765.7" y="211" font-size="10" fill="#68737f">120ms · ran</text><text x="176" y="240" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">escalate</text><rect x="756.7" y="227" width="113.3" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>escalate.page — ran, 120.2ms</title></rect><text x="879.0" y="241" font-size="10" fill="#68737f">120ms · ran</text><text x="176" y="270" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">file</text><rect x="870.0" y="257" width="2.5" height="19" rx="4" fill="#68737f" opacity=".78" stroke="#68737f" stroke-width="1"><title>file.queue — skipped, 0.0ms — not taken — an upstream branch went the other way</title></rect><text x="881.5" y="271" font-size="10" fill="#68737f">0ms · skipped</text><rect x="190" y="304" width="10" height="10" rx="2" fill="#c0392b" opacity=".78"/><text x="205" y="313" font-size="10" fill="#68737f">failed</text> <rect x="286" y="304" width="10" height="10" rx="2" fill="#68737f" opacity=".78"/><text x="301" y="313" font-size="10" fill="#68737f">skipped</text> <rect x="382" y="304" width="10" height="10" rx="2" fill="#2d6cb5" opacity=".78"/><text x="397" y="313" font-size="10" fill="#68737f">cached</text> <rect x="478" y="304" width="10" height="10" rx="2" fill="#c98a2b" opacity=".78"/><text x="493" y="313" font-size="10" fill="#68737f">fell back</text> <rect x="574" y="304" width="10" height="10" rx="2" fill="#1f8a4c" opacity=".78"/><text x="589" y="313" font-size="10" fill="#68737f">ran</text></svg>', title='one worker — the same plan, run sequentially', note='Bars are placed at the time each step began.', width=1000, height=336)

In [7]:
viz.timeline(two, title='two workers — topic and urgency overlap')

Figure(svg='<svg viewBox="0 0 1000 336" width="1000" height="336" style="max-width:none" role="img"><text x="190" y="34" font-size="10.5" fill="#68737f">0s</text><text x="870" y="34" text-anchor="end" font-size="10.5" fill="#68737f">0.602s</text><line x1="190" y1="42" x2="870" y2="42" stroke="#dfe5ec" stroke-width="1"/><text x="176" y="90" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">read</text><rect x="190.2" y="77" width="135.8" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>read.jsonl — ran, 120.2ms</title></rect><text x="335.0" y="91" font-size="10" fill="#68737f">120ms · ran</text><text x="176" y="120" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">topic</text><rect x="326.3" y="107" width="135.7" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>topic.embed — ran, 120.2ms</title></rect><text x="471.1" y="121" font-size="10" fill="#68737f">120ms · ran</text><text x="176" y="150" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">urgency</text><rect x="326.6" y="137" width="135.7" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>urgency.rules — ran, 120.1ms</title></rect><text x="471.3" y="151" font-size="10" fill="#68737f">120ms · ran</text><text x="176" y="180" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">enrich</text><rect x="462.5" y="167" width="135.7" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>enrich.history — ran, 120.2ms</title></rect><text x="607.2" y="181" font-size="10" fill="#68737f">120ms · ran</text><text x="176" y="210" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">route</text><rect x="598.3" y="197" width="135.7" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>route.decide — ran, 120.2ms</title></rect><text x="743.0" y="211" font-size="10" fill="#68737f">120ms · ran</text><text x="176" y="240" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">escalate</text><rect x="734.3" y="227" width="135.7" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>escalate.page — ran, 120.2ms</title></rect><text x="879.0" y="241" font-size="10" fill="#68737f">120ms · ran</text><text x="176" y="270" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">file</text><rect x="734.5" y="257" width="2.5" height="19" rx="4" fill="#68737f" opacity=".78" stroke="#68737f" stroke-width="1"><title>file.queue — skipped, 0.0ms — not taken — an upstream branch went the other way</title></rect><text x="746.0" y="271" font-size="10" fill="#68737f">0ms · skipped</text><rect x="190" y="304" width="10" height="10" rx="2" fill="#c0392b" opacity=".78"/><text x="205" y="313" font-size="10" fill="#68737f">failed</text> <rect x="286" y="304" width="10" height="10" rx="2" fill="#68737f" opacity=".78"/><text x="301" y="313" font-size="10" fill="#68737f">skipped</text> <rect x="382" y="304" width="10" height="10" rx="2" fill="#2d6cb5" opacity=".78"/><text x="397" y="313" font-size="10" fill="#68737f">cached</text> <rect x="478" y="304" width="10" height="10" rx="2" fill="#c98a2b" opacity=".78"/><text x="493" y="313" font-size="10" fill="#68737f">fell back</text> <rect x="574" y="304" width="10" height="10" rx="2" fill="#1f8a4c" opacity=".78"/><text x="589" y="313" font-size="10" fill="#68737f">ran</text></svg>', title='two workers — topic and urgency overlap', note='Bars are placed at the time each step began; 0.721s of work in 0.602s of clock.', width=1000, height=336)

Same plan, same graph, same route. The only difference is `workers=2`, and the
picture is where you can see it: `topic` and `urgency` start at the same moment
instead of one after the other. Nothing about the graph changed to allow that —
the independence was already in the wiring, and one number decided whether to
use it.

Look at `file` too. It is grey, meaning skipped: the branch went to `escalate`,
so `file` never had to run. That is not a failure and it is not a success. It is
a third thing, and merging it into either would misreport what happened.

One more run, to show the last colour. `escalate.page` declares the effect
`notify.person` — it pages a human. Turn effects off and it is refused *before*
it runs.

In [8]:
refused = execute.run(plan, runtime, workers=2, allow_effects=False, strict=False)
print("ok:", refused.ok, " stopped at:", refused.stopped_at)
viz.timeline(refused, title="effects off — the paging step is refused, not run")

ok: False  stopped at: escalate


Figure(svg='<svg viewBox="0 0 1000 336" width="1000" height="336" style="max-width:none" role="img"><text x="190" y="34" font-size="10.5" fill="#68737f">0s</text><text x="870" y="34" text-anchor="end" font-size="10.5" fill="#68737f">0.482s</text><line x1="190" y1="42" x2="870" y2="42" stroke="#dfe5ec" stroke-width="1"/><text x="176" y="90" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">read</text><rect x="190.1" y="77" width="169.6" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>read.jsonl — ran, 120.2ms</title></rect><text x="368.7" y="91" font-size="10" fill="#68737f">120ms · ran</text><text x="176" y="120" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">topic</text><rect x="360.1" y="107" width="169.5" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>topic.embed — ran, 120.2ms</title></rect><text x="538.6" y="121" font-size="10" fill="#68737f">120ms · ran</text><text x="176" y="150" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">urgency</text><rect x="360.3" y="137" width="169.5" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>urgency.rules — ran, 120.1ms</title></rect><text x="538.8" y="151" font-size="10" fill="#68737f">120ms · ran</text><text x="176" y="180" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">enrich</text><rect x="530.0" y="167" width="169.6" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>enrich.history — ran, 120.2ms</title></rect><text x="708.6" y="181" font-size="10" fill="#68737f">120ms · ran</text><text x="176" y="210" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">route</text><rect x="699.7" y="197" width="169.7" height="19" rx="4" fill="#1f8a4c" opacity=".78" stroke="#1f8a4c" stroke-width="1"><title>route.decide — ran, 120.3ms</title></rect><text x="878.4" y="211" font-size="10" fill="#68737f">120ms · ran</text><text x="176" y="240" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">escalate</text><rect x="869.9" y="227" width="2.5" height="19" rx="4" fill="#c0392b" opacity=".78" stroke="#c0392b" stroke-width="1"><title>escalate.page — failed, 0.0ms — refused: this step declares notify.person and effects are off</title></rect><text x="881.4" y="241" font-size="10" fill="#68737f">0ms · failed</text><text x="176" y="270" text-anchor="end" font-size="11" font-weight="600" fill="#22303f">file</text><rect x="870.0" y="257" width="2.5" height="19" rx="4" fill="#68737f" opacity=".78" stroke="#68737f" stroke-width="1"><title>file.queue — skipped, 0.0ms — not taken — an upstream branch went the other way</title></rect><text x="881.5" y="271" font-size="10" fill="#68737f">0ms · skipped</text><rect x="190" y="304" width="10" height="10" rx="2" fill="#c0392b" opacity=".78"/><text x="205" y="313" font-size="10" fill="#68737f">failed</text> <rect x="286" y="304" width="10" height="10" rx="2" fill="#68737f" opacity=".78"/><text x="301" y="313" font-size="10" fill="#68737f">skipped</text> <rect x="382" y="304" width="10" height="10" rx="2" fill="#2d6cb5" opacity=".78"/><text x="397" y="313" font-size="10" fill="#68737f">cached</text> <rect x="478" y="304" width="10" height="10" rx="2" fill="#c98a2b" opacity=".78"/><text x="493" y="313" font-size="10" fill="#68737f">fell back</text> <rect x="574" y="304" width="10" height="10" rx="2" fill="#1f8a4c" opacity=".78"/><text x="589" y="313" font-size="10" fill="#68737f">ran</text></svg>', title='effects off — the paging step is refused, not run', note='Bars are placed at the time each step began; 0.601s of work in 0.482s of clock.', width=1000, height=336)

Declaring the effect is what makes that possible. A step that quietly sends an
email is indistinguishable from one that does not until the email arrives.

## 5 · Scoreboard — what did the winner beat

`solve` tries routes, runs them, judges the output and keeps the best plus a
fallback. The champion on its own is a number with no denominator. This is the
denominator.

In [9]:
from browsergraph.solve import solve

# The judge looks at the *output*, not at whether the code threw. Without this,
# "it worked" means "it did not raise" — and a route that returns nothing
# passes that test with full marks.
def judge(run):
    tickets = run.output("read")
    return bool(tickets), float(len(tickets))

answer = solve(bench, runtime, verify=judge, attempts=6,
               inputs=None, workspace=str(WORK), seed=3)

print(answer.text(bench))
viz.scoreboard(answer)

champion scored 2.0000 (4 of 6 tried worked, out of 24 possible)
    read             read.jsonl
    topic            topic.keywords
    urgency          urgency.rules
    enrich           enrich.none
    route            route.decide
    escalate         escalate.page
    file             file.queue
  fallback 1: differs at topic
  2 did not work; first reason: ran, but the output was not acceptable
  4.02s


Figure(svg='<svg viewBox="0 0 1000 312" width="1000" height="312" style="max-width:none" role="img"><text x="236" y="42" text-anchor="end" font-size="10.5" fill="#68737f">route</text><text x="250" y="42" font-size="10.5" fill="#68737f">score</text><line x1="250" y1="50" x2="850" y2="50" stroke="#dfe5ec" stroke-width="1"/><text x="236" y="93" text-anchor="end" font-size="11" fill="#22303f">★ champion</text><rect x="250" y="79" width="600.0" height="20" rx="4" fill="#1f8a4c" opacity=".74" stroke="#1f8a4c" stroke-width="1"><title>read=read.jsonl, topic=topic.keywords, urgency=urgency.rules, enrich=enrich.none, route=route.decide, escalate=escalate.page, file=file.queue</title></rect><text x="860.0" y="94" font-size="10.5" fill="#68737f">2 · 602ms</text><text x="236" y="125" text-anchor="end" font-size="11" fill="#22303f">↳ topic.embed</text><rect x="250" y="111" width="600.0" height="20" rx="4" fill="#c98a2b" opacity=".74" stroke="#c98a2b" stroke-width="1"><title>read=read.jsonl, topic=topic.embed, urgency=urgency.rules, enrich=enrich.none, route=route.decide, escalate=escalate.page, file=file.queue</title></rect><text x="860.0" y="126" font-size="10.5" fill="#68737f">2 · 602ms</text><text x="236" y="157" text-anchor="end" font-size="11" fill="#22303f">enrich.history</text><rect x="250" y="143" width="600.0" height="20" rx="4" fill="#2d6cb5" opacity=".74" stroke="#2d6cb5" stroke-width="1"><title>read=read.jsonl, topic=topic.keywords, urgency=urgency.rules, enrich=enrich.history, route=route.decide, escalate=escalate.page, file=file.queue</title></rect><text x="860.0" y="158" font-size="10.5" fill="#68737f">2 · 722ms</text><text x="236" y="189" text-anchor="end" font-size="11" fill="#22303f">urgency.model</text><rect x="250" y="175" width="600.0" height="20" rx="4" fill="#2d6cb5" opacity=".74" stroke="#2d6cb5" stroke-width="1"><title>read=read.jsonl, topic=topic.keywords, urgency=urgency.model, enrich=enrich.none, route=route.decide, escalate=escalate.page, file=file.queue</title></rect><text x="860.0" y="190" font-size="10.5" fill="#68737f">2 · 602ms</text><text x="236" y="221" text-anchor="end" font-size="11" fill="#22303f">read.csv, topic.bagofwords, urgenc</text><rect x="250" y="207" width="2.5" height="20" rx="4" fill="#c0392b" opacity=".74" stroke="#c0392b" stroke-width="1"><title>read=read.csv, topic=topic.bagofwords, urgency=urgency.model, enrich=enrich.history, route=route.decide, escalate=escalate.page, file=file.queue — ran, but the output was not acceptable</title></rect><text x="262.5" y="222" font-size="10.5" fill="#68737f">did not work · 722ms</text><text x="236" y="253" text-anchor="end" font-size="11" fill="#22303f">read.csv, urgency.model, enrich.hi</text><rect x="250" y="239" width="2.5" height="20" rx="4" fill="#c0392b" opacity=".74" stroke="#c0392b" stroke-width="1"><title>read=read.csv, topic=topic.keywords, urgency=urgency.model, enrich=enrich.history, route=route.decide, escalate=escalate.page, file=file.queue — ran, but the output was not acceptable</title></rect><text x="262.5" y="254" font-size="10.5" fill="#68737f">did not work · 722ms</text></svg>', title='4 of 6 routes worked', note='Out of 24 possible. Labels name only what differs from the champion.', width=1000, height=312)

`read.csv` returns an empty list. It never raises, so it is not a crash — and
every route using it scores zero because the judge looked at the output. That
is the whole argument for keeping judging separate from running, in one picture.

## 6 · Evidence — which step is the problem

Solving left something behind: a store of what each candidate did, every time it
ran. That is what this last chart is drawn from.

For each step of the champion, it asks one question — *is this the right pick,
and by how much?* — and answers in signed bits:

```
bits = log2(how often my pick worked / how often its best rival worked)
```

Positive means the choice is beating every alternative the evidence has seen.
Negative means something else is doing better and this is the step to change.

In [10]:
from browsergraph.evidence import per_step_bits, stages_of

bits = per_step_bits(answer.evidence, answer.champion, stages_of(bench))
for stage, value in bits.items():
    print(f"  {stage:<10} {value:+.2f} bits")

viz.evidence(bits, title="the champion, step by step")

  read       +1.74 bits
  topic      +0.00 bits
  urgency    +1.00 bits
  enrich     +1.00 bits
  route      +0.00 bits
  escalate   +0.00 bits
  file       +0.00 bits


Figure(svg='<svg viewBox="0 0 940 308" width="940" height="308" style="max-width:none" role="img"><line x1="525.0" y1="44" x2="525.0" y2="278" stroke="#dfe5ec" stroke-width="1"/><text x="525.0" y="294" text-anchor="middle" font-size="9.5" fill="#68737f">0 bits</text><text x="184" y="73" text-anchor="end" font-size="11" fill="#22303f">read</text><rect x="525.0" y="60" width="325.0" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="858.0" y="73" font-size="10" text-anchor="start" fill="#68737f">+1.74</text><text x="184" y="103" text-anchor="end" font-size="11" fill="#22303f">topic</text><rect x="525.0" y="90" width="1.5" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="533.0" y="103" font-size="10" text-anchor="start" fill="#68737f">+0.00</text><text x="184" y="133" text-anchor="end" font-size="11" fill="#22303f">urgency</text><rect x="525.0" y="120" width="187.1" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="720.1" y="133" font-size="10" text-anchor="start" fill="#68737f">+1.00</text><text x="184" y="163" text-anchor="end" font-size="11" fill="#22303f">enrich</text><rect x="525.0" y="150" width="187.1" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="720.1" y="163" font-size="10" text-anchor="start" fill="#68737f">+1.00</text><text x="184" y="193" text-anchor="end" font-size="11" fill="#22303f">route</text><rect x="525.0" y="180" width="1.5" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="533.0" y="193" font-size="10" text-anchor="start" fill="#68737f">+0.00</text><text x="184" y="223" text-anchor="end" font-size="11" fill="#22303f">escalate</text><rect x="525.0" y="210" width="1.5" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="533.0" y="223" font-size="10" text-anchor="start" fill="#68737f">+0.00</text><text x="184" y="253" text-anchor="end" font-size="11" fill="#22303f">file</text><rect x="525.0" y="240" width="1.5" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="533.0" y="253" font-size="10" text-anchor="start" fill="#68737f">+0.00</text></svg>', title='the champion, step by step', note='Positive: this step supported the route. Negative: it argued against it.', width=940, height=308)

Three things to read off this, and the third one is the important one.

**A step with one candidate sits at zero, and should.** There was no choice, so
there is nothing to be right or wrong about. A number there would invite reading
meaning into a decision nobody made.

**`read` is strongly positive.** `read.jsonl` produced tickets and `read.csv`
produced none, every time. That is a real difference and the evidence found it.

**`topic` comes out slightly negative — and that is noise, not a finding.** Look
at the judge: it counts tickets, and the topic step cannot change how many
tickets there are. So no amount of running will ever tell you which topic
candidate is better; the small number is just which routes happened to get
tried. A chart cannot know that. You have to.

The rule that falls out: **a step your judge cannot see is a step your evidence
cannot rank.** If it matters, measure it — and if you are not measuring it, do
not read its bar.

## 7 · A figure, for somewhere that is not a web page

The same route space as matplotlib, because you cannot paste an SVG into a
LaTeX document without a conversion step.

In [11]:
import matplotlib
matplotlib.use("Agg")

ax = viz.to_figure(bench, route=after, alternative=before)
ax.figure.savefig(WORK / "route-space.png", dpi=120, bbox_inches="tight")
print("wrote", WORK / "route-space.png")
ax.figure

wrote work/route-space.png


<Figure size 1540x202 with 1 Axes>

## Everything at once

`viz.report` writes all of it to one self-contained page — no CDN, no fonts, no
fetch, so it opens from a `file://` URL on a machine with no network. That is
the only kind of artefact worth committing next to the code that made it.

In [12]:
page = viz.write_report(bench, WORK / "triage.html", route=after, alternative=before,
                        search=[("every route", found.total),
                                ("policy-eligible", found.eligible_total),
                                ("scored", found.examined), ("chosen", 1)],
                        bits=bits, run=two, solution=answer)
text = pathlib.Path(page).read_text()
print(f"{page}  ({len(text):,} bytes)")
for forbidden in ("http://", "https://", "<script src", "@import"):
    print(f"  reaches for {forbidden!r}: {forbidden in text}")

work/triage.html  (28,192 bytes)
  reaches for 'http://': False
  reaches for 'https://': False
  reaches for '<script src': False
  reaches for '@import': False


## And the same graph, as text

Two other renderers, for the places a picture will not go. Mermaid renders in a
GitHub README, where inline SVG does not. JSON is for a front end that would
rather draw it itself — emitting it costs nothing and removes the argument that
adopting the format means adopting this renderer.

In [13]:
print(viz.to_mermaid(bench, after))

graph LR
  read["Read the tickets<br/><i>read.jsonl</i>"]
  topic["Work out the topic<br/><i>topic.embed</i>"]
  urgency["Work out the urgency<br/><i>urgency.rules</i>"]
  enrich["Add what we know<br/><i>enrich.history</i>"]
  route{{"Escalate or file?<br/><i>route.decide</i><br/>[branch]"}}
  escalate["Page somebody<br/><i>escalate.page</i>"]
  file["Put it in the queue<br/><i>file.queue</i>"]
  read --> topic
  read --> urgency
  topic --> enrich
  enrich -->|topic| route
  urgency -->|urgency| route
  route -->|escalate| escalate
  route -->|file| file
  style read stroke:#c0392b,stroke-width:2px
  style topic stroke:#c0392b,stroke-width:2px
  style urgency stroke:#c0392b,stroke-width:2px
  style enrich stroke:#c0392b,stroke-width:2px
  style route stroke:#c0392b,stroke-width:2px
  style escalate stroke:#c0392b,stroke-width:2px
  style file stroke:#c0392b,stroke-width:2px


Notice `route{{...}}` — Mermaid's rhombus, because the step is a branch — and
`enrich[...]` as an ordinary box. The shape carries the meaning in whichever
renderer you are using.

## What the pictures are for

Not decoration. Each one is a claim that can be checked:

* the **shape** says two steps are independent, and the layering proves it;
* the **route space** says how many options there were, and counts them;
* the **funnel** says how many were examined, and does not round it up;
* the **evidence** chart says which step to look at, and signs it;
* the **timeline** says whether the parallel plan really ran in parallel;
* the **scoreboard** says what the winner beat, failures included;
* the **figure** says the same thing somewhere a browser is not.

A diagram that cannot be wrong is a diagram that is not saying anything.

## Draw your own

Every function here takes a graph and nothing else. There is no browser code in
the drawing, no ticket code, nothing about this example:

```python
from browsergraph import viz

viz.dag(bench)                       # any workbench
viz.route_space(bench, route=r)      # any route
viz.timeline(run)                    # any finished run
viz.scoreboard(answer)               # any solve result
viz.write_report(bench, "out.html", route=r, run=run, solution=answer)
```

If your problem can be written as steps with typed ports, it can be drawn, and
no drawing code changes. That is the whole claim, and this notebook is the test
of it — the pictures above are of support tickets, and not one line of the
renderer knows what a ticket is.